In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "modules" / "minio"))

from minio_spark import MinioSparkClient
from os import getenv
from pyspark.sql import functions as F

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
client = MinioSparkClient(
    "https://minio.fdi.ucm.es",
    getenv("MINIO_ACCESS_KEY"),
    getenv("MINIO_SECRET_KEY"),
    "pd2/cityenjoyer",
    memory = 8,
    heapsize = 4,
    num_part = 500
)
client.connect()

:: loading settings :: url = jar:file:/home/alvaro/Documentos/PROYECTO%20DE%20DATOS%202/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/alvaro/.ivy2.5.2/cache
The jars for the packages stored in: /home/alvaro/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4d200a57-8031-422e-950c-a9871e49a842;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
:: resolution report :: resolve 221ms :: artifacts dl 11ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.1.3.Final from central in [default]
	software.amazon.awssdk#bundle;2.24.6 from central in [default]
	--------------------------------------

In [3]:
df = client.read_parquet("21-25_merged.parquet")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [4]:
df.show(5)

+--------+-------------------+-------------------+-------------+------------+------------+------------+----------+------------+-----------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|trip_distance|PULocationID|DOLocationID|payment_type|tip_amount|tolls_amount|fare_amount|total_amount|
+--------+-------------------+-------------------+-------------+------------+------------+------------+----------+------------+-----------+------------+
|       2|2022-08-01 00:03:30|2022-08-01 00:14:08|         2.59|          35|          61|           7|         0|           0|       1315|        1428|
|       2|2022-08-01 00:31:21|2022-08-01 01:08:01|         7.59|          65|          39|           7|         0|           0|       2896|        3146|
|       2|2022-08-01 00:26:14|2022-08-01 00:39:20|         2.49|         188|          89|           7|         0|           0|       1317|        1431|
|       2|2022-08-01 00:02:24|2022-08-01 00:23:37|         9.21|         237|     

In [6]:
columnas_num = ["trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount"]
stats_para_graficar = []

# 1. Tomamos solo el 1% del dataset. 
# Esto reduce 1.300 millones de filas a ~13 millones. Spark lo procesará en segundos.
print("Extrayendo muestra del 1%...")
df_sample = df.sample(False, 0.01, seed=42)

# Cacheamos la muestra en memoria distribuida para que el bucle no tenga que recalcularla 5 veces
df_sample.cache()

for col in columnas_num:
    print(f"Calculando métricas para: {col}...")
    
    # 2. Calculamos cuartiles sobre LA MUESTRA. 
    # Al ser solo 13 millones de filas, podemos pedir más precisión (0.001) sin penalizar el tiempo.
    cuantiles = df_sample.approxQuantile(col, [0.25, 0.5, 0.75], 0.001)
    q1, med, q3 = cuantiles[0], cuantiles[1], cuantiles[2]
    
    iqr = q3 - q1
    limite_inf = q1 - (1.5 * iqr)
    limite_sup = q3 + (1.5 * iqr)
    
    # Buscamos los extremos reales dentro de esa muestra del 1%
    min_max = df_sample.agg(F.min(col).alias("min"), F.max(col).alias("max")).collect()[0]
    min_real = min_max["min"]
    max_real = min_max["max"]
    
    # Guardamos el resumen estadístico
    stats_para_graficar.append({
        'label': col,
        'med': med,
        'q1': q1,
        'q3': q3,
        'whislo': max(min_real, limite_inf),
        'whishi': min(max_real, limite_sup),
        'fliers': [min_real, max_real] # Mostramos los valores extremos absolutos de la muestra
    })

# Liberamos la memoria caché de Spark
df_sample.unpersist()

print("Dibujando el gráfico...")

# 3. Matplotlib dibuja las cajas instantáneamente usando solo los resúmenes matemáticos
fig, ax = plt.subplots(figsize=(14, 7))

ax.bxp(stats_para_graficar, showfliers=True, patch_artist=True, 
       boxprops=dict(facecolor='lightblue', color='black'))

# Escala logarítmica simétrica (symlog) para poder visualizar valores gigantes sin aplastar la caja
plt.yscale('symlog') 
plt.title("Boxplots de variables numéricas (Basado en muestra del 1%)")
plt.ylabel("Valor (Escala Logarítmica)")
plt.grid(axis='y', linestyle='--', alpha=0.6)

Calculando métricas del boxplot para: trip_distance...


ERROR:root:KeyboardInterrupt while sending command.               (0 + 5) / 146]
Traceback (most recent call last):
  File "/home/alvaro/Documentos/PROYECTO DE DATOS 2/C-ity-enjoyers/.venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/alvaro/Documentos/PROYECTO DE DATOS 2/C-ity-enjoyers/.venv/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/home/alvaro/snap/code/220/.local/share/uv/python/cpython-3.13.11-linux-x86_64-gnu/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/home/alvaro/Documentos/PROYECTO DE DATOS 2/C-ity-enjoyers/.venv/lib/python3.13/site

KeyboardInterrupt: 